# 01 · Análisis exploratorio de PaySim

Ejecutable de arriba abajo (`make eda`; requiere `make data`). Solo datos agregados: no se muestra ninguna fila
cruda. Al final escribe `ml/reports/eda.md` y los gráficos de `ml/reports/figures/`.
No calcula features ni entrena modelos, y no decide los cortes ni el conjunto de features A o B.

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats

ROOT = Path.cwd().resolve().parents[1]  # el cuaderno se ejecuta desde ml/notebooks
CFG = yaml.safe_load((ROOT / "ml/config.yaml").read_text())
FIG = ROOT / "ml/reports/figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {"figure.figsize": (6.4, 2.6), "figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3}
)

df = pd.read_csv(
    ROOT / "data/raw/paysim.csv",
    dtype={"type": "category", "nameOrig": "category", "nameDest": "category"},
)
df["day"] = (df["step"] - 1) // 24 + 1
R = {}  # cifras que irán a eda.md
R["rows"] = len(df)
R["fraud"] = int(df["isFraud"].sum())
R["prev"] = df["isFraud"].mean()
R["steps"] = int(df["step"].max())
R["unique_steps"] = int(df["step"].nunique())
print(f"filas={R['rows']:,} fraudes={R['fraud']:,} prevalencia={R['prev']:.4%} pasos={R['steps']}")

filas=6,362,620 fraudes=8,213 prevalencia=0.1291% pasos=743


## 1. Filas, fraudes y prevalencia

In [2]:
def summarize(g):
    out = g.agg(rows=("isFraud", "size"), fraud=("isFraud", "sum"))
    out["prevalence_%"] = (100 * out["fraud"] / out["rows"]).round(4)
    return out


by_type = summarize(df.groupby("type", observed=True))
by_day = summarize(df.groupby("day"))
by_step = summarize(df.groupby("step"))
R["by_type"] = by_type
display(by_type)
display(by_day.T)

fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
ax[0].bar(by_day.index, by_day["fraud"])
ax[0].set(title="Fraudes por día (24 pasos)", xlabel="día")
ax[1].plot(by_step.index, by_step["prevalence_%"], lw=0.6)
ax[1].set(title="Prevalencia por paso (%)", xlabel="step")
fig.tight_layout()
fig.savefig(FIG / "fraude_por_dia_y_paso.png", dpi=80)
plt.show()
R["fraud_days_zero"] = int((by_day["fraud"] == 0).sum())
R["fraud_per_day_min"], R["fraud_per_day_max"] = (
    int(by_day["fraud"].min()),
    int(by_day["fraud"].max()),
)
R["prev_step_max"] = by_step["prevalence_%"].max()
R["prev_step_min"] = by_step["prevalence_%"].min()

,rows,fraud,prevalence_%
type,,,
CASH_IN,1399284,0,0.0000
CASH_OUT,2237500,4116,0.1840
DEBIT,41432,0,0.0000
PAYMENT,2151495,0,0.0000
TRANSFER,532909,4097,0.7688


day,1,2,3,4,5,6,7,8,9,10,...,22,23,24,25,26,27,28,29,30,31
rows,574255.0000,455238.0000,1070.000,28240.0000,9789.0000,441005.0000,420583.0000,449637.0000,417919.000,392945.0000,...,53437.0000,51012.0000,32709.000,57853.0000,13885.0000,8578.0000,14661.0000,54890.0000,11287.0000,272.0
fraud,271.0000,309.0000,310.000,262.0000,252.0000,228.0000,272.0000,278.0000,255.000,282.0000,...,256.0000,216.0000,280.000,240.0000,272.0000,280.0000,248.0000,260.0000,268.0000,272.0
prevalence_%,0.0472,0.0679,28.972,0.9278,2.5743,0.0517,0.0647,0.0618,0.061,0.0718,...,0.4791,0.4234,0.856,0.4148,1.9589,3.2642,1.6916,0.4737,2.3744,100.0


/tmp/ipykernel_58422/2054406097.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. ¿El fraude solo ocurre en TRANSFER y CASH_OUT?

In [3]:
fraud_types = df.loc[df["isFraud"] == 1, "type"].value_counts()
only_two = set(fraud_types[fraud_types > 0].index) == {"TRANSFER", "CASH_OUT"}
R["fraud_types"] = fraud_types.to_dict()
R["fraud_only_transfer_cashout"] = only_two
print(
    fraud_types.to_dict(),
    "->",
    "solo TRANSFER y CASH_OUT" if only_two else "HAY FRAUDE EN OTROS TIPOS",
)
cash_in = int((df["type"] == "CASH_IN").sum())
R["cash_in_rows"] = cash_in
print(
    f"CASH_IN (fuera de config.types): {cash_in:,} filas, {int(df.loc[df.type == 'CASH_IN', 'isFraud'].sum())} fraudes"
)

{'CASH_OUT': 4116, 'TRANSFER': 4097, 'CASH_IN': 0, 'DEBIT': 0, 'PAYMENT': 0} -> solo TRANSFER y CASH_OUT
CASH_IN (fuera de config.types): 1,399,284 filas, 0 fraudes


## 3. Volumen por paso: ¿es estacionario?

In [4]:
vol = df.groupby("step").size()
hour = (vol.index - 1) % 24
hod = vol.groupby(hour).mean()
rho, p_rho = stats.spearmanr(vol.index, vol.values)
slope = stats.linregress(vol.index, vol.values)
h1, h2 = vol[vol.index <= R["steps"] // 2], vol[vol.index > R["steps"] // 2]
mw = stats.mannwhitneyu(h1, h2)
day_mean = vol.groupby((vol.index - 1) // 24 + 1).mean()
R["vol"] = {
    "mean": vol.mean(),
    "std": vol.std(),
    "cv": vol.std() / vol.mean(),
    "min": int(vol.min()),
    "max": int(vol.max()),
    "rho": rho,
    "p_rho": p_rho,
    "slope": slope.slope,
    "h1": h1.mean(),
    "h2": h2.mean(),
    "p_mw": mw.pvalue,
    "hod_ratio": hod.max() / hod.min(),
    "first_day": day_mean.iloc[0],
    "last_day": day_mean.iloc[-1],
    "first_steps": vol.iloc[:12].tolist(),
}
print(
    {
        k: (round(v, 4) if isinstance(v, float) else v)
        for k, v in R["vol"].items()
        if k != "first_steps"
    }
)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
ax[0].plot(vol.index, vol.values, lw=0.6)
ax[0].set(title="Transacciones por paso", xlabel="step", yscale="log")
ax[1].bar(hod.index, hod.values)
ax[1].set(title="Volumen medio por hora del día", xlabel="hora (step-1) mod 24")
fig.tight_layout()
fig.savefig(FIG / "volumen_por_paso.png", dpi=80)
plt.show()

{'mean': np.float64(8563.4186), 'std': np.float64(13388.6932), 'cv': np.float64(1.5635), 'min': 2, 'max': 51352, 'rho': np.float64(-0.2757), 'p_rho': np.float64(0.0), 'slope': np.float64(-23.9388), 'h1': np.float64(14168.1213), 'h2': np.float64(2973.7823), 'p_mw': np.float64(0.0), 'hod_ratio': np.float64(522.0097), 'first_day': np.float64(23927.2917), 'last_day': np.float64(11.8261)}


/tmp/ipykernel_58422/4156398911.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Orígenes y destinos

In [5]:
orig_counts = df["nameOrig"].value_counts()
R["orig_unique"] = len(orig_counts)
R["orig_once_of_unique"] = (orig_counts == 1).mean()
R["orig_once_of_rows"] = (orig_counts == 1).sum() / len(df)
dest_counts = df["nameDest"].value_counts()
R["dest_unique"] = len(dest_counts)
R["dest_once_of_unique"] = (dest_counts == 1).mean()
R["dest_reused_rows"] = dest_counts[dest_counts > 1].sum() / len(df)
R["dest_p"] = dest_counts.quantile([0.5, 0.9, 0.99]).to_dict()
R["dest_max"] = int(dest_counts.max())
is_merchant = df["nameDest"].astype(str).str.startswith("M")
R["dest_merchant_rows"] = is_merchant.mean()
print({k: v for k, v in R.items() if k.startswith(("orig_", "dest_"))})

# Destinos de la partición de prueba que ya aparecen en la de entrenamiento (tipos de config.types).
sp = CFG["split"]
in_types = df["type"].isin(CFG["types"])
tr = df[in_types & df["step"].between(*sp["train"])]
te = df[in_types & df["step"].between(*sp["test"])]
seen = set(tr["nameDest"].unique())
te_seen = te["nameDest"].isin(seen)
te_unique = te["nameDest"].drop_duplicates()
tef = te[te["isFraud"] == 1]
R["test_dest_seen_rows"] = te_seen.mean()
R["test_dest_seen_unique"] = te_unique.isin(seen).mean()
R["test_fraud_dest_seen"] = tef["nameDest"].isin(seen).mean() if len(tef) else float("nan")
R["test_nonfraud_dest_seen"] = te.loc[te["isFraud"] == 0, "nameDest"].isin(seen).mean()
# Misma medida con destinos no comerciantes ("C...") para quitar el efecto de los comercios.
te_c = te[~te["nameDest"].astype(str).str.startswith("M")]
R["test_dest_seen_rows_nonmerchant"] = te_c["nameDest"].isin(seen).mean()
print({k: round(v, 4) for k, v in R.items() if k.startswith("test_")})

{'orig_unique': 6353307, 'orig_once_of_unique': np.float64(0.998536510198547), 'orig_once_of_rows': np.float64(0.9970749471129817), 'dest_unique': 2722362, 'dest_once_of_unique': np.float64(0.8311547105050687), 'dest_reused_rows': np.float64(0.644375430247288), 'dest_p': {0.5: 1.0, 0.9: 5.0, 0.99: 25.0}, 'dest_max': 113, 'dest_merchant_rows': np.float64(0.3381460781879163)}


{'test_dest_seen_rows': np.float64(0.4066), 'test_dest_seen_unique': np.float64(0.394), 'test_fraud_dest_seen': np.float64(0.2986), 'test_nonfraud_dest_seen': np.float64(0.4083), 'test_dest_seen_rows_nonmerchant': np.float64(0.731)}


## 5. Fuga en los saldos y qué marca `isFlaggedFraud`

In [6]:
f = df[df["isFraud"] == 1]
l = df[df["isFraud"] == 0]
tol = 0.01


def frac(mask):
    return float(mask.mean())


leak = {
    "fraude: newbalanceOrig == 0": frac(f["newbalanceOrig"] == 0),
    "legítimo: newbalanceOrig == 0": frac(l["newbalanceOrig"] == 0),
    "fraude: oldbalanceOrg == amount (vacía la cuenta)": frac(
        (f["oldbalanceOrg"] - f["amount"]).abs() < tol
    ),
    "legítimo: oldbalanceOrg == amount": frac((l["oldbalanceOrg"] - l["amount"]).abs() < tol),
    "fraude: newbalanceOrig == oldbalanceOrg - amount": frac(
        (f["newbalanceOrig"] - (f["oldbalanceOrg"] - f["amount"])).abs() < tol
    ),
    "fraude: oldbalanceDest == 0": frac(f["oldbalanceDest"] == 0),
    "legítimo: oldbalanceDest == 0": frac(l["oldbalanceDest"] == 0),
    "fraude: newbalanceDest == oldbalanceDest + amount": frac(
        (f["newbalanceDest"] - (f["oldbalanceDest"] + f["amount"])).abs() < tol
    ),
    "fraude: oldbalanceOrg == 0": frac(f["oldbalanceOrg"] == 0),
    "legítimo: oldbalanceOrg == 0": frac(l["oldbalanceOrg"] == 0),
}
# Comparación justa: solo TRANSFER y CASH_OUT, los únicos tipos con fraude.
tc = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]
tcf, tcl = tc[tc["isFraud"] == 1], tc[tc["isFraud"] == 0]
leak["TRANSFER/CASH_OUT legítimo: newbalanceOrig == 0"] = frac(tcl["newbalanceOrig"] == 0)
leak["TRANSFER/CASH_OUT legítimo: oldbalanceDest == 0"] = frac(tcl["oldbalanceDest"] == 0)
leak["TRANSFER/CASH_OUT legítimo: oldbalanceOrg == amount"] = frac(
    (tcl["oldbalanceOrg"] - tcl["amount"]).abs() < tol
)
R["leak"] = leak
display(pd.Series(leak).round(4).to_frame("fracción"))

flag = df[df["isFlaggedFraud"] == 1]
over = df[(df["type"] == "TRANSFER") & (df["amount"] > 200_000)]
R["flag"] = {
    "n": len(flag),
    "types": flag["type"].value_counts().to_dict(),
    "fraud_in_flag": int(flag["isFraud"].sum()),
    "min_amount": float(flag["amount"].min()) if len(flag) else None,
    "captured": float(flag["isFraud"].sum() / R["fraud"]),
    "transfer_over_200k": len(over),
    "transfer_over_200k_flagged": int(over["isFlaggedFraud"].sum()),
    "transfer_over_200k_fraud": int(over["isFraud"].sum()),
    "flagged_with_orig_balance_zero": int((flag["oldbalanceOrg"] == 0).sum()) if len(flag) else 0,
}
print(R["flag"])

,fracción
fraude: newbalanceOrig == 0,0.9805
legítimo: newbalanceOrig == 0,0.5668
fraude: oldbalanceOrg == amount (vacía la cuenta),0.9782
legítimo: oldbalanceOrg == amount,0.0000
fraude: newbalanceOrig == oldbalanceOrg - amount,0.9945
fraude: oldbalanceDest == 0,0.6515
legítimo: oldbalanceDest == 0,0.4248
fraude: newbalanceDest == oldbalanceDest + amount,0.4350
fraude: oldbalanceOrg == 0,0.0050
legítimo: oldbalanceOrg == 0,0.3309


{'n': 16, 'types': {'TRANSFER': 16, 'CASH_IN': 0, 'CASH_OUT': 0, 'DEBIT': 0, 'PAYMENT': 0}, 'fraud_in_flag': 16, 'min_amount': 353874.22, 'captured': 0.0019481310118105442, 'transfer_over_200k': 409110, 'transfer_over_200k_flagged': 16, 'transfer_over_200k_fraud': 2740, 'flagged_with_orig_balance_zero': 0}


## 6. Fraudes por partición (cortes de `config.yaml`) y por hueco

In [7]:
sp = CFG["split"]
parts = {
    "train": tuple(sp["train"]),
    "hueco 1": (sp["train"][1] + 1, sp["valid"][0] - 1),
    "valid": tuple(sp["valid"]),
    "hueco 2": (sp["valid"][1] + 1, sp["test"][0] - 1),
    "test": tuple(sp["test"]),
}
rows = []
for name, (a, b) in parts.items():
    m = df[df["step"].between(a, b)]
    mt = m[m["type"].isin(CFG["types"])]
    rows.append(
        dict(
            partición=name,
            pasos=f"{a}-{b}",
            filas=len(m),
            fraudes=int(m["isFraud"].sum()),
            **{"prevalencia_%": round(100 * m["isFraud"].mean(), 4)},
            fraudes_en_config_types=int(mt["isFraud"].sum()),
        )
    )
part_table = pd.DataFrame(rows)
display(part_table)
R["part_table"] = part_table
R["gap_ok"] = all(parts[k][1] - parts[k][0] + 1 == sp["gap_steps"] for k in ("hueco 1", "hueco 2"))
R["min_fraud_part"] = int(
    part_table.loc[part_table["partición"].isin(["train", "valid", "test"]), "fraudes"].min()
)
print(
    "hueco = gap_steps:",
    R["gap_ok"],
    "| mínimo de fraudes en una partición usada:",
    R["min_fraud_part"],
)
assert R["min_fraud_part"] >= 500, "alguna partición tiene menos de 500 fraudes: revisar los cortes"

fig, ax = plt.subplots(figsize=(6.4, 2.6))
cum = by_step["fraud"]
ax.bar(cum.index, cum.values, width=1.0, color="0.6")
for (name, (a, b)), c in zip(parts.items(), ["C0", "0.9", "C1", "0.9", "C2"]):
    ax.axvspan(a - 0.5, b + 0.5, color=c, alpha=0.25, lw=0)
    ax.text((a + b) / 2, ax.get_ylim()[1] * 0.9, name, ha="center", fontsize=7)
ax.set(title="Fraudes por paso y particiones", xlabel="step")
fig.tight_layout()
fig.savefig(FIG / "particiones.png", dpi=80)
plt.show()

,partición,pasos,filas,fraudes,prevalencia_%,fraudes_en_config_types
0,train,1-400,5787030,4477,0.0774,4477
1,hueco 1,401-424,209359,308,0.1471,308
2,valid,425-550,169630,1280,0.7546,1280
3,hueco 2,551-574,34290,286,0.8341,286
4,test,575-744,162311,1862,1.1472,1862


hueco = gap_steps: True | mínimo de fraudes en una partición usada: 1280


/tmp/ipykernel_58422/2427877428.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Informe `ml/reports/eda.md`

In [8]:
def pct(x, d=2):
    return f"{100 * x:.{d}f} %"


def table(d):
    d = d.reset_index() if d.index.name else d
    cols = list(d.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in d.iterrows():
        lines.append(
            "| "
            + " | ".join(f"{v:,}" if isinstance(v, (int, np.integer)) else str(v) for v in r)
            + " |"
        )
    return "\n".join(lines)


v, fl, lk = R["vol"], R["flag"], R["leak"]
md_text = f"""# EDA de PaySim

Generado por `ml/notebooks/01_eda.ipynb` (`make eda`). Solo cifras agregadas. No decide los cortes ni el
conjunto de features A o B: son decisiones pendientes.

## Resumen

- {R["rows"]:,} filas, {R["unique_steps"]} pasos distintos de 1 h (step máximo {R["steps"]}; la ficha dice 744), {R["fraud"]:,} fraudes (prevalencia {pct(R["prev"], 3)}).
- Fraude solo en TRANSFER y CASH_OUT: **{"sí" if R["fraud_only_transfer_cashout"] else "NO"}** ({R["fraud_types"]}).
- Fraudes por día: entre {R["fraud_per_day_min"]} y {R["fraud_per_day_max"]}; días sin fraude: {R["fraud_days_zero"]}.
- Volumen por paso **{"no " if abs(v["rho"]) > 0.3 or v["p_mw"] < 0.01 else ""}estacionario** en tendencia (ver §Volumen).
- Orígenes que aparecen una sola vez: {pct(R["orig_once_of_unique"])} de los orígenes distintos.
- Destinos de la prueba ya vistos en entrenamiento: {pct(R["test_dest_seen_rows"])} de las filas.
- Fuga de saldos: en los fraudes `newbalanceOrig` es 0 en el {pct(lk["fraude: newbalanceOrig == 0"])} y la cuenta se vacía
  (`oldbalanceOrg == amount`) en el {pct(lk["fraude: oldbalanceOrg == amount (vacía la cuenta)"])}.
- `isFlaggedFraud` marca {fl["n"]} filas y captura {fl["fraud_in_flag"]} fraudes ({pct(fl["captured"], 3)} del total).

## Fraudes por partición (cortes de `ml/config.yaml`)

{table(R["part_table"])}

Los huecos miden {CFG["split"]["gap_steps"]} pasos ({"correcto" if R["gap_ok"] else "REVISAR"}). Mínimo de fraudes en una partición
usada: {R["min_fraud_part"]:,} (umbral de §5c: 500).

![particiones](figures/particiones.png)

## Filas, fraudes y prevalencia por tipo

{table(R["by_type"])}

CASH_IN ({R["cash_in_rows"]:,} filas) no figura en `config.types`.

![fraude por día y por paso](figures/fraude_por_dia_y_paso.png)

La prevalencia por paso es inestable (pasos de madrugada con pocas filas): ver el gráfico; la de cada partición está en la tabla de arriba.

## Volumen por paso

Media {v["mean"]:,.0f} transacciones/paso, desviación {v["std"]:,.0f} (CV {v["cv"]:.2f}), mínimo {v["min"]:,}, máximo {v["max"]:,}.
Spearman paso–volumen: ρ = {v["rho"]:.3f} (p = {v["p_rho"]:.2g}); pendiente {v["slope"]:.2f} tx/paso por paso.
Mitad 1 frente a mitad 2: {v["h1"]:,.0f} frente a {v["h2"]:,.0f} (Mann–Whitney p = {v["p_mw"]:.2g}).
Primer día {v["first_day"]:,.0f}/paso, último {v["last_day"]:,.0f}/paso. Razón máx/mín de la hora del día: {v["hod_ratio"]:.1f}.
Primeros 12 pasos: {v["first_steps"]}.

![volumen por paso](figures/volumen_por_paso.png)

## Orígenes y destinos

- Orígenes distintos: {R["orig_unique"]:,}; aparecen una sola vez: {pct(R["orig_once_of_unique"])} (sobre las filas: {pct(R["orig_once_of_rows"])}).
- Destinos distintos: {R["dest_unique"]:,}; una sola vez: {pct(R["dest_once_of_unique"])}; filas cuyo destino se repite: {pct(R["dest_reused_rows"])}.
  Cuantiles de apariciones por destino (p50/p90/p99): {R["dest_p"]}; máximo {R["dest_max"]:,}.
  Filas con destino comerciante (`M...`): {pct(R["dest_merchant_rows"])}.
- Partición de prueba (tipos de `config.types`), fracción cuyo destino aparece en entrenamiento:
  filas {pct(R["test_dest_seen_rows"])}; destinos distintos {pct(R["test_dest_seen_unique"])};
  solo destinos no comerciantes {pct(R["test_dest_seen_rows_nonmerchant"])}.
  En filas de fraude {pct(R["test_fraud_dest_seen"])}; en filas legítimas {pct(R["test_nonfraud_dest_seen"])}.

## Fuga en saldos

{table(pd.Series({k: round(x, 4) for k, x in lk.items()}).rename_axis("medida").to_frame("fracción"))}

`isFlaggedFraud`: {fl["n"]} filas marcadas (tipos {fl["types"]}), importe mínimo {fl["min_amount"]}.
{fl["fraud_in_flag"]} son fraude: captura {pct(fl["captured"], 3)} de los fraudes. Hay {fl["transfer_over_200k"]:,} TRANSFER con
`amount > 200.000`, de las que {fl["transfer_over_200k_flagged"]} están marcadas y {fl["transfer_over_200k_fraud"]:,} son fraude.
Es una regla de importe, no una etiqueta: prohibida como entrada (`contracts/features.yaml`).
"""
(ROOT / "ml/reports/eda.md").write_text(md_text)
print(md_text[:1500])

# EDA de PaySim

Generado por `ml/notebooks/01_eda.ipynb` (`make eda`). Solo cifras agregadas. No decide los cortes ni el
conjunto de features A o B: son decisiones pendientes.

## Resumen

- 6,362,620 filas, 743 pasos distintos de 1 h (step máximo 743; la ficha dice 744), 8,213 fraudes (prevalencia 0.129 %).
- Fraude solo en TRANSFER y CASH_OUT: **sí** ({'CASH_OUT': 4116, 'TRANSFER': 4097, 'CASH_IN': 0, 'DEBIT': 0, 'PAYMENT': 0}).
- Fraudes por día: entre 216 y 320; días sin fraude: 0.
- Volumen por paso **no estacionario** en tendencia (ver §Volumen).
- Orígenes que aparecen una sola vez: 99.85 % de los orígenes distintos.
- Destinos de la prueba ya vistos en entrenamiento: 40.66 % de las filas.
- Fuga de saldos: en los fraudes `newbalanceOrig` es 0 en el 98.05 % y la cuenta se vacía
  (`oldbalanceOrg == amount`) en el 97.82 %.
- `isFlaggedFraud` marca 16 filas y captura 16 fraudes (0.195 % del total).

## Fraudes por partición (cortes de `ml/config.yaml`)

| partición | pasos | fila